# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

> **Lane:** Ranking Signal Analysis · **Card:** ML-04 · **Dataset:** the full warehouse release
> (`FlyRank/internship-warehouse`, build v20260703).
>
> Iteration month is **`month=2026-03`** — mid-panel on purpose. The `_sample` table is the
> final month (June 2026), which is the natural outcome window of any past→future label, so it
> is never used here. June stays sealed.

### Setup

Set `HF_TOKEN` in Colab's Secrets panel (🔑) or paste it at the prompt. Never type a token into
a cell — this repo is public.

In [1]:
%pip install -q duckdb pandas scikit-learn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os, getpass, pathlib, time, json
import duckdb, pandas as pd, numpy as np

def _token():
    if os.environ.get("HF_TOKEN"):
        return os.environ["HF_TOKEN"]
    try:
        from google.colab import userdata
        if userdata.get("HF_TOKEN"):
            return userdata.get("HF_TOKEN")
    except Exception:
        pass
    cached = pathlib.Path.home() / ".cache/huggingface/token"
    if cached.exists() and cached.read_text().strip():
        return cached.read_text().strip()
    return getpass.getpass("Paste your Hugging Face READ token (hf_...): ")

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute("CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN ?)", [_token()])

B = "hf://datasets/FlyRank/internship-warehouse"
MONTH = "2026-03"                      # mid-panel iteration month
LABEL_MONTH = "2026-04"                # the month my label is measured in
FACT  = f"read_parquet('{B}/fact_content_daily_performance/month={MONTH}/*.parquet')"
LABEL = f"read_parquet('{B}/fact_content_daily_performance/month={LABEL_MONTH}/*.parquet')"

REPO = pathlib.Path.cwd()
for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
    if (p / "data/raw/content_refresh_anonymized.csv").exists():
        REPO = p; break
OUT = REPO / "work/outputs"; OUT.mkdir(parents=True, exist_ok=True)
rel = lambda q: q.relative_to(REPO).as_posix()

def cached(name, sql):
    f = OUT / f"w03_{name}.parquet"
    if f.exists():
        return pd.read_parquet(f)
    t = time.time(); df = con.sql(sql).df(); df.to_parquet(f, index=False)
    print(f"[scanned {name} in {time.time()-t:.0f}s -> cached]")
    return df

print("connected · iteration month:", MONTH, "· label month:", LABEL_MONTH)

connected · iteration month: 2026-03 · label month: 2026-04


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

### The contract in plain words — answers 1, 2 and 3 of 5

**1. What one row means for my lane.**
One row is **one pseudonymized content item (a page), evaluated once**. The warehouse fact table
is at `report_date × client × content` grain — one row per page per *day* — which is not my unit.
I collapse those daily rows into one row per page by aggregating over a feature window, then
attach an outcome measured in a strictly later window.

**2. Which table(s) I'll use.**
`fact_content_daily_performance` (the daily grain, aggregated into windows), joined to
`dim_content` for static page metadata. `dim_clients` is read only to understand coverage, never
joined into features. `fact_content_query_90d` is **not** used at all — see answer 5.

**3. Which time window.**

| Window | Dates | Role |
|---|---|---|
| Feature window | **`month=2026-03`** (this notebook) | everything knowable at the decision moment |
| Outcome window | **`month=2026-04`** | where the label is measured |
| Sealed test | label month **2026-06** | untouched until the very end |

The decision moment is **2026-03-31**. Features end there; the label starts the next day. The two
do not overlap by a single day.

In [3]:
# A one-line orientation check before any claim: am I pointed at the month I think I am?
span = cached("v2_orientation", f"""
    SELECT '{MONTH}' AS iteration_month,
           MIN(report_date) AS first_day, MAX(report_date) AS last_day,
           COUNT(*) AS daily_rows
    FROM {FACT}
""")
print(span.to_string(index=False))
print("\nDecision moment = 2026-03-31. Label is measured in 2026-04. No overlap.")

[scanned v2_orientation in 5s -> cached]
iteration_month  first_day   last_day  daily_rows
        2026-03 2026-03-01 2026-03-31     9841378

Decision moment = 2026-03-31. Label is measured in 2026-04. No overlap.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### The contract in plain words — answers 4 and 5 of 5

**4. What I'd predict or rank (label / proxy).**

> **`is_position_decline`** — 1 when a page's **impression-weighted average search position**
> gets worse by **1.0 place or more** between the feature window and the outcome month.

Impression-weighted means `SUM(gsc_sum_position) / SUM(gsc_impressions)`, never the mean of daily
`gsc_avg_position` — averaging daily figures would let a 2-impression day count as much as a
20,000-impression one. It is a **proxy**: nobody records "this page declined", so I define decline
from observed position movement and say so plainly.

I use it to **rank**, not to classify — the output is a review queue ordered by risk.

**5. One thing I deliberately exclude.**

> **The entire `fact_content_query_90d` table.**

Its window is fixed at **2026-04-02 → 2026-06-30**, which *starts inside my outcome month*. Query
mix (diversity, concentration, the rare/anonymized tail) would probably be the strongest feature
family available to this lane — and I cannot touch any of it, because every column is measured
partly during the period I am trying to predict. Even the `*_prev30` columns sit after my
2026-03-31 decision moment.

This is settled by a date comparison, not by judgement. Query 3 below is where I check it.

**Everything else in the four buckets, briefly:** features are feature-window ranking behaviour
plus static `dim_content` metadata; the label block is `o_pos` / `pos_delta` /
`is_position_decline`; context is the hash IDs and dates (grouping and splitting only, never
features); also excluded are `trend_direction`/`trend_pct` (label-derived), and
`last_optimized_date`/`optimization_eligible_date` (decision-derived — they record what FlyRank's
own system already chose to do, so learning them means learning the old rule rather than the world).

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Exactly **three** verification queries, each proving one sentence I wrote above.

In [4]:
# ---- QUERY 1 of 3 --- GRAIN: one row really is report_date x client x content ----
q1 = cached("v2_q1_grain", f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS n
    FROM {FACT}
    GROUP BY 1,2,3
    HAVING COUNT(*) > 1
    LIMIT 5
""")
print("QUERY 1 — grain probe (duplicate keys should be zero rows)\n")
print(f"duplicate (report_date, client, content) keys found: {len(q1)}")
print("VERDICT: grain holds — one row per page per day, exactly as the contract states."
      if q1.empty else q1.to_string(index=False))

[scanned v2_q1_grain in 24s -> cached]
QUERY 1 — grain probe (duplicate keys should be zero rows)

duplicate (report_date, client, content) keys found: 0
VERDICT: grain holds — one row per page per day, exactly as the contract states.


In [5]:
# ---- QUERY 2 of 3 --- MY SLICE: row count and date span ----
q2 = cached("v2_q2_slice", f"""
    SELECT COUNT(*)                        AS daily_rows,
           COUNT(DISTINCT content_hash_id) AS content_items,
           COUNT(DISTINCT client_hash_id)  AS clients,
           MIN(report_date)                AS span_start,
           MAX(report_date)                AS span_end
    FROM {FACT}
""")
print(f"QUERY 2 — my slice ({MONTH}): row count and date span\n")
print(q2.to_string(index=False))
r = q2.iloc[0]
print(f"\n{int(r.daily_rows):,} daily rows covering {int(r.content_items):,} content items "
      f"across {int(r.clients)} clients,")
print(f"spanning {r.span_start} to {r.span_end} — a complete calendar month, mid-panel.")

[scanned v2_q2_slice in 3s -> cached]
QUERY 2 — my slice (2026-03): row count and date span

 daily_rows  content_items  clients span_start   span_end
    9841378         331437       55 2026-03-01 2026-03-31

9,841,378 daily rows covering 331,437 content items across 55 clients,
spanning 2026-03-01 00:00:00 to 2026-03-31 00:00:00 — a complete calendar month, mid-panel.


In [6]:
# ---- QUERY 3 of 3 --- AVAILABILITY: filter with IS TRUE, count survivors ----
# The flags are THREE-valued (TRUE / FALSE / NULL). `= TRUE` and `NOT ...` both mishandle NULL,
# so the contract commits to IS TRUE / IS NOT TRUE everywhere.
q3 = cached("v2_q3_availability", f"""
    SELECT
        COUNT(*)                                                      AS all_rows,
        COUNT(*) FILTER (WHERE gsc_data_available IS TRUE)            AS gsc_is_true,
        COUNT(*) FILTER (WHERE gsc_data_available IS NULL)            AS gsc_is_null,
        COUNT(*) FILTER (WHERE ga4_data_available IS TRUE)            AS ga4_is_true,
        COUNT(*) FILTER (WHERE ga4_data_available IS NULL)            AS ga4_is_null,
        COUNT(*) FILTER (WHERE gsc_data_available IS TRUE
                           AND gsc_impressions > 0)                   AS my_usable_rows
    FROM {FACT}
""")
print(f"QUERY 3 — availability in {MONTH}, filtered with IS TRUE\n")
a = q3.iloc[0]
for k in q3.columns:
    print(f"  {k:16s} {int(a[k]):>12,}")
print(f"\nRows surviving my filter (gsc_data_available IS TRUE AND gsc_impressions > 0): "
      f"{int(a.my_usable_rows):,}")
print(f"That is {a.my_usable_rows/a.all_rows:.1%} of the month.")
print(f"\nWhy IS TRUE and not = TRUE: {int(a.ga4_is_null):,} rows carry a NULL "
      f"ga4_data_available ({a.ga4_is_null/a.all_rows:.1%} of the month) —")
print("neither zero-filled nor flagged FALSE. `= FALSE` or `NOT ...` silently drops them.")

[scanned v2_q3_availability in 13s -> cached]
QUERY 3 — availability in 2026-03, filtered with IS TRUE

  all_rows            9,841,378
  gsc_is_true         3,611,061
  gsc_is_null                 0
  ga4_is_true           413,966
  ga4_is_null         3,018,741
  my_usable_rows      3,611,061

Rows surviving my filter (gsc_data_available IS TRUE AND gsc_impressions > 0): 3,611,061
That is 36.7% of the month.

Why IS TRUE and not = TRUE: 3,018,741 rows carry a NULL ga4_data_available (30.7% of the month) —
neither zero-filled nor flagged FALSE. `= FALSE` or `NOT ...` silently drops them.


### The five features — and when each is knowable

Five, maximum, all aggregated from **`month=2026-03`** only. The decision moment is
**2026-03-31**; each line below says why the feature exists on or before that date.

In [7]:
# One query builds the five-feature frame from 2026-03, with the label from 2026-04.
frame = cached("v2_feature_frame", f"""
WITH f AS (
    SELECT content_hash_id,
           ANY_VALUE(client_hash_id)                                     AS client_hash_id,
           SUM(gsc_sum_position) / NULLIF(SUM(gsc_impressions),0)        AS f_pos,
           SUM(gsc_impressions)                                          AS f_impressions,
           100.0*SUM(gsc_clicks) / NULLIF(SUM(gsc_impressions),0)        AS f_ctr,
           STDDEV_SAMP(gsc_avg_position) FILTER (WHERE gsc_impressions>0) AS f_pos_volatility,
           COUNT(DISTINCT report_date) FILTER (WHERE gsc_impressions>0)  AS f_days_with_impressions
    FROM {FACT}
    WHERE gsc_data_available IS TRUE
    GROUP BY content_hash_id
),
o AS (
    SELECT content_hash_id,
           SUM(gsc_impressions)                                   AS o_impressions,
           SUM(gsc_sum_position)/NULLIF(SUM(gsc_impressions),0)   AS o_pos
    FROM {LABEL}
    WHERE gsc_data_available IS TRUE
    GROUP BY content_hash_id
)
SELECT f.*, o.o_pos,
       o.o_pos - f.f_pos AS pos_delta,
       CASE WHEN o.o_pos - f.f_pos >= 1.0 THEN 1 ELSE 0 END AS is_position_decline
FROM f JOIN o USING (content_hash_id)
WHERE f.f_impressions >= 100 AND o.o_impressions >= 30
""")
FEATURES = ["f_pos","f_impressions","f_ctr","f_pos_volatility","f_days_with_impressions"]
print(f"feature frame: {len(frame):,} pages · {frame.client_hash_id.nunique()} clients · "
      f"{len(FEATURES)} features")
print(f"label base rate: {frame.is_position_decline.mean():.4f}\n")
print(frame[FEATURES].describe().round(2).to_string())

[scanned v2_feature_frame in 59s -> cached]
feature frame: 98,404 pages · 42 clients · 5 features
label base rate: 0.5179

          f_pos  f_impressions     f_ctr  f_pos_volatility  f_days_with_impressions
count  98404.00       98404.00  98404.00          98381.00                 98404.00
mean      14.20        2825.21      0.26              6.97                    28.65
std       14.50        7037.22      0.42              6.45                     4.89
min        0.02         100.00      0.00              0.01                     1.00
25%        4.74         303.00      0.00              2.12                    29.00
50%        8.18         833.00      0.13              4.82                    31.00
75%       18.86        2616.00      0.37              9.81                    31.00
max      106.89      617124.00     15.58             91.69                    31.00


| # | Feature | Available when? |
|---|---|---|
| 1 | `f_pos` — impression-weighted average position in March | **Knowable at the decision moment because** it is computed only from March's daily rows; the last one lands 2026-03-31, the day I decide. |
| 2 | `f_impressions` — total March impressions | **Knowable at the decision moment because** it is a sum over days that have already happened; nothing after 2026-03-31 contributes. |
| 3 | `f_ctr` — March clicks ÷ March impressions × 100 | **Knowable at the decision moment because** both the numerator and denominator are totals of the same closed window; it is a ratio of two facts already on the books. |
| 4 | `f_pos_volatility` — standard deviation of daily position within March | **Knowable at the decision moment because** it describes the *shape* of a series that has already finished — instability I can see, not instability I am assuming. |
| 5 | `f_days_with_impressions` — count of March days with ≥1 impression | **Knowable at the decision moment because** it counts days inside the window; the maximum it can reach is 31, which is the month I just observed. |

Every one of them stops at **2026-03-31**. The label starts **2026-04-01**. The gap between those
two dates is the entire safety margin of this project.

### The trap — one label-derived column, on purpose

Now I break my own contract deliberately. `o_pos` is April's position — the column the label is
computed from. It is exactly the kind of field that looks like just another number in a dataframe.

In [8]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score

y  = frame["is_position_decline"].values
gp = frame["client_hash_id"].values

def quick_score(cols, label):
    """Same model, same grouped split, same metric. Only the column list changes."""
    p = np.zeros(len(frame))
    for tr, te in GroupKFold(n_splits=3).split(frame, y, groups=gp):
        m = RandomForestClassifier(n_estimators=120, min_samples_leaf=5,
                                   n_jobs=-1, random_state=20260808)
        m.fit(frame.iloc[tr][cols].fillna(-1), y[tr])
        p[te] = m.predict_proba(frame.iloc[te][cols].fillna(-1))[:, 1]
    auc = roc_auc_score(y, p)
    print(f"{label:44s} AUC = {auc:.4f}")
    return auc

print(f"base rate = {y.mean():.4f}   (AUC 0.500 = no skill, 1.000 = perfect)\n")
honest = quick_score(FEATURES, "5 honest features")
leaked = quick_score(FEATURES + ["o_pos"], "5 features + o_pos (LABEL-DERIVED)")
print(f"\njump: +{leaked - honest:.4f} AUC")

base rate = 0.5179   (AUC 0.500 = no skill, 1.000 = perfect)



5 honest features                            AUC = 0.5576


5 features + o_pos (LABEL-DERIVED)           AUC = 0.9988

jump: +0.4413 AUC


In [9]:
# Delete the leak and keep the honest number. This is the point of the exercise.
LEAKY = ["o_pos", "pos_delta"]
MODEL_FEATURES = [c for c in FEATURES if c not in LEAKY]
assert not (set(MODEL_FEATURES) & set(LEAKY)), "a label-derived column survived the cleanup"

print("removed :", LEAKY)
print("kept    :", MODEL_FEATURES)
print(f"\nHONEST NUMBER I CARRY FORWARD: AUC = {honest:.4f}")
print(f"(the leaked {leaked:.4f} is discarded — it is the answer, read back to me)")

json.dump({"iteration_month": MONTH, "label_month": LABEL_MONTH,
           "unit": "one content item, evaluated once",
           "tables": ["fact_content_daily_performance", "dim_content"],
           "label_rule": "is_position_decline = (o_pos - f_pos) >= 1.0",
           "excluded_headline": "fact_content_query_90d (window opens 2026-04-02, inside the outcome month)",
           "features": FEATURES,
           "population": "f_impressions >= 100 AND o_impressions >= 30",
           "rows": int(len(frame)), "clients": int(frame.client_hash_id.nunique()),
           "base_rate": round(float(y.mean()), 4),
           "leak_demo": {"honest_auc": round(float(honest), 4),
                         "leaked_auc": round(float(leaked), 4),
                         "leaked_column": "o_pos"}},
          open(OUT / "w03_contract_receipt.json", "w"), indent=2)
print(f"\nreceipt -> {rel(OUT / 'w03_contract_receipt.json')}")

removed : ['o_pos', 'pos_delta']
kept    : ['f_pos', 'f_impressions', 'f_ctr', 'f_pos_volatility', 'f_days_with_impressions']

HONEST NUMBER I CARRY FORWARD: AUC = 0.5576
(the leaked 0.9988 is discarded — it is the answer, read back to me)

receipt -> work/outputs/w03_contract_receipt.json


**What just happened.** Adding one column took the score from roughly *modest* to *near-perfect*.
No error was raised, no warning printed — the dataframe simply had one more numeric column in it,
and the model quietly read the answer.

This is the notebook-02 lesson performed on real warehouse data: **leakage does not look like a
bug, it looks like success.** A near-perfect score on a messy real-world problem is not a
triumph, it is a symptom — and the correct response is to go looking for the column that is
secretly the label rather than to celebrate.

The honest number stays. `o_pos` and `pos_delta` are label-side columns and live in the label
bucket of the contract, never the feature bucket.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

### One named limitation of my slice

> **The query table's window makes query-mix features unusable for this label — and the panel has
> no later period to escape into.**

`fact_content_query_90d` covers **2026-04-02 → 2026-06-30**. My outcome month is April 2026, so
the table's window opens *inside* the period I am predicting. Query diversity, concentration, and
the rare/anonymized tail are almost certainly the richest signals available for a lane called
Ranking Signal Analysis — and every one of them is off-limits here.

The obvious fix is to move the label to a month after 2026-06-30 so the query window sits safely
in the past. **The panel ends on 2026-06-30**, so that month does not exist. The limitation is
structural, not a choice I can engineer around, and it means my feature set is narrower than the
warehouse makes it look.

*(Other limits — the portfolio-wide downward position drift inside the label, the unbalanced
panel, and the outcome-window information in my population filter — are carried in the capstone's
Limitations section.)*

In [10]:
# The limitation, verified rather than asserted.
qw = cached("v2_query_window", f"""
    SELECT MIN(window_start) AS window_start, MAX(window_end) AS window_end, COUNT(*) AS rows
    FROM read_parquet('{B}/fact_content_query_90d.parquet')
""")
print(qw.to_string(index=False))
ws = pd.to_datetime(qw.window_start.iloc[0])
print(f"\ndecision moment      : 2026-03-31")
print(f"outcome window       : 2026-04-01 .. 2026-04-30")
print(f"query table window   : {ws.date()} .. {pd.to_datetime(qw.window_end.iloc[0]).date()}")
assert ws > pd.Timestamp("2026-03-31")
print("\nThe query window opens AFTER my decision moment and inside my outcome month.")
print("Excluded — by a date check, not a judgement call.")

[scanned v2_query_window in 3s -> cached]
window_start window_end    rows
  2026-04-02 2026-06-30 2414248

decision moment      : 2026-03-31
outcome window       : 2026-04-01 .. 2026-04-30
query table window   : 2026-04-02 .. 2026-06-30

The query window opens AFTER my decision moment and inside my outcome month.
Excluded — by a date check, not a judgement call.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.